In [ ]:
import pandas as pd
import requests as rq
from pathlib import Path

PROJECT_PATH = Path.cwd() #return the current working directory of the project
BRONZE_PATH = PROJECT_PATH / "Data" / "Bronze"
WEATHER_BRONZE_PATH = BRONZE_PATH / "Weather"

In [ ]:
csv_files = list(WEATHER_BRONZE_PATH.glob("*.csv"))

df_input = pd.concat(
    [pd.read_csv(file) for file in csv_files],
    ignore_index=True
) if csv_files else pd.DataFrame()

df_input

,ID,FLIGHT_DATE,ORIGIN,ORIGIN_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WEATHER_DELAY,LATITUDE,LONGITUDE
0,1,2019-01-01,ABQ,"Albuquerque, NM",525,NaN,NaN,35.022248,-106.630602
1,2,2019-01-01,MFR,"Medford, OR",600,NaN,NaN,42.386218,-122.879185
2,3,2019-01-01,GCK,"Garden City, KS",608,NaN,NaN,37.930893,-100.733276
3,4,2019-01-01,PSP,"Palm Springs, CA",615,NaN,NaN,33.835167,-116.509668
4,5,2019-01-01,GFK,"Grand Forks, ND",620,NaN,NaN,47.952392,-97.195101


In [ ]:
# Define the URL of the API
base_url = "https://archive-api.open-meteo.com/v1/archive"


# Initialise the lists
responses = []
dataframes = []


# Make API calls
for index, row in df_input.iterrows():

    # Convert the date to format YYYY-MM-DD
    flight_date = pd.to_datetime(row["FLIGHT_DATE"]).strftime("%Y-%m-%d")

    requested_latitude = float(row["LATITUDE"])
    requested_longitude = float(row["LONGITUDE"])

    params = {
        "latitude": requested_latitude,
        "longitude": requested_longitude,
        "start_date": flight_date,
        "end_date": flight_date,
        "hourly": (
            "temperature_2m,rain,snowfall,snow_depth,"
            "wind_speed_10m,wind_gusts_10m"
        )
    }

    response = rq.get(base_url, params=params, timeout=30)
    response.raise_for_status()

    # Transform the HTTP response into a Python dictionnary
    data = response.json()

    # Keep the full answer in the list
    responses.append(data)

    # Transform the hourly data into a DataFrame
    hourly_df = pd.DataFrame(data["hourly"])

    # Add the coordinates
    hourly_df["requested_latitude"] = requested_latitude
    hourly_df["requested_longitude"] = requested_longitude
    hourly_df["latitude"] = data["latitude"]
    hourly_df["longitude"] = data["longitude"]

    dataframes.append(hourly_df)

# Merge all data into a unique DataFrame
df = pd.concat(dataframes, ignore_index=True)

# Convert the dates into a datetime format 
df["time"] = pd.to_datetime(df["time"])

df['date'] = df['time'].dt.date
df['hour'] = df['time'].dt.strftime('%H%M')

# Organise the columns
columns = [
    "requested_latitude",
    "requested_longitude",
    "latitude",
    "longitude",
    "date",
    "hour",
    "temperature_2m",
    "rain",
    "snowfall",
    "snow_depth",
    "wind_speed_10m",
    "wind_gusts_10m",
]

df = df[columns]

# Verify the result
display(df)
print("Dimensions du DataFrame :", df.shape)
print("Nombre de réponses API :", len(responses))



,requested_latitude,requested_longitude,latitude,longitude,date,hour,temperature_2m,rain,snowfall,snow_depth,wind_speed_10m,wind_gusts_10m
0,35.022248,-106.630602,35.043934,-106.603060,2019-01-01,0000,-1.2,0.0,0.07,0.07,6.4,7.2
1,35.022248,-106.630602,35.043934,-106.603060,2019-01-01,0100,-1.9,0.0,0.14,0.07,5.8,9.7
2,35.022248,-106.630602,35.043934,-106.603060,2019-01-01,0200,0.1,0.0,0.07,0.07,2.6,8.6
3,35.022248,-106.630602,35.043934,-106.603060,2019-01-01,0300,-2.3,0.0,0.00,0.07,6.5,9.7
4,35.022248,-106.630602,35.043934,-106.603060,2019-01-01,0400,-3.8,0.0,0.00,0.07,7.6,11.9
...,...,...,...,...,...,...,...,...,...,...,...,...
115,47.952392,-97.195101,47.978910,-97.176056,2019-01-01,1900,-21.3,0.0,0.00,0.16,11.2,19.8
116,47.952392,-97.195101,47.978910,-97.176056,2019-01-01,2000,-20.5,0.0,0.00,0.16,12.2,22.0
117,47.952392,-97.195101,47.978910,-97.176056,2019-01-01,2100,-20.1,0.0,0.00,0.16,14.1,24.8
118,47.952392,-97.195101,47.978910,-97.176056,2019-01-01,2200,-20.1,0.0,0.00,0.16,15.6,25.9


Dimensions du DataFrame : (120, 12)
Nombre de réponses API : 5


In [23]:
# Rename the columns to fit into the database table schema
df = df.rename(columns={
    "hour": "HOUR_ID",
    "date": "WEATHER_DATE",
    "temperature_2m": "TEMPERATURE_2M",
    "rain": "RAIN",
    "snowfall": "SNOWFALL_CM",
    "snow_depth": "SNOW_DEPTH_M",
    "wind_speed_10m": "WIND_SPEED_10M",
    "wind_gusts_10m": "WIND_GUSTS_10M",
    "requested_latitude": "LATITUDE",
    "requested_longitude": "LONGITUDE"
})

# Select the columns and place them into the right order
df = df[[
    "HOUR_ID",
    "WEATHER_DATE",
    "TEMPERATURE_2M",
    "RAIN",
    "SNOWFALL_CM",
    "SNOW_DEPTH_M",
    "WIND_SPEED_10M",
    "WIND_GUSTS_10M",
    "LATITUDE",
    "LONGITUDE"
]]


df

,HOUR_ID,WEATHER_DATE,TEMPERATURE_2M,RAIN,SNOWFALL_CM,SNOW_DEPTH_M,WIND_SPEED_10M,WIND_GUSTS_10M,LATITUDE,LONGITUDE
0,0000,2019-01-01,-1.2,0.0,0.07,0.07,6.4,7.2,35.022248,-106.630602
1,0100,2019-01-01,-1.9,0.0,0.14,0.07,5.8,9.7,35.022248,-106.630602
2,0200,2019-01-01,0.1,0.0,0.07,0.07,2.6,8.6,35.022248,-106.630602
3,0300,2019-01-01,-2.3,0.0,0.00,0.07,6.5,9.7,35.022248,-106.630602
4,0400,2019-01-01,-3.8,0.0,0.00,0.07,7.6,11.9,35.022248,-106.630602
...,...,...,...,...,...,...,...,...,...,...
115,1900,2019-01-01,-21.3,0.0,0.00,0.16,11.2,19.8,47.952392,-97.195101
116,2000,2019-01-01,-20.5,0.0,0.00,0.16,12.2,22.0,47.952392,-97.195101
117,2100,2019-01-01,-20.1,0.0,0.00,0.16,14.1,24.8,47.952392,-97.195101
118,2200,2019-01-01,-20.1,0.0,0.00,0.16,15.6,25.9,47.952392,-97.195101


In [24]:
# Export the DataFrame

df.to_csv("output API_clean.csv", index=False)

print("Export done !")

Export done !
